# 05. Image-Swap Experiment

This notebook runs the controlled image-swap experiment to test model robustness.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').absolute()))

import torch
import numpy as np
import pandas as pd

from src.utils import set_seed, get_device, ensure_dir
from src.dataset import MultimodalFakeNewsDataset, Sample
from src.preprocessing import MultimodalPreprocessor
from src.classifier import MultimodalClassifier, ClassifierConfig
from src.image_swap import ImageSwapExperiment, SwapConfig, controlled_swap_evaluation
from src.evaluate import Evaluator
from torch.utils.data import DataLoader

## Configuration

In [ ]:
set_seed(42)
device = get_device()

DATA_DIR = Path('data/processed')
IMAGES_DIR = Path('data/fakeddit/images')
CHECKPOINT_DIR = Path('artifacts/checkpoints')
RESULTS_DIR = Path('results')
ensure_dir(RESULTS_DIR)

## Load Test Data

In [ ]:
test_df = pd.read_csv(DATA_DIR / 'test.csv')

def create_samples(df, images_dir):
    samples = []
    for _, row in df.iterrows():
        text = str(row.get('clean_title', row.iloc[0]))
        image_id = str(row.get('image_id', ''))
        label = int(row.get('binary_label', row.iloc[-1]))
        
        image_path = images_dir / f"{image_id}.jpg" if image_id else None
        
        sample = Sample(
            sample_id=str(row.name),
            text=text,
            image_path=str(image_path) if image_path and image_path.exists() else None,
            label=label,
            metadata={}
        )
        samples.append(sample)
    return samples

test_samples = create_samples(test_df, IMAGES_DIR)
test_dataset = MultimodalFakeNewsDataset(test_samples)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Test dataset: {len(test_dataset)} samples")

## Load Models

In [ ]:
preprocessor = MultimodalPreprocessor(device=device)

# Load consistency-aware model
consistency_checkpoint = CHECKPOINT_DIR / 'consistency_aware' / 'consistency_aware_best.pt'
if consistency_checkpoint.exists():
    checkpoint = torch.load(consistency_checkpoint, map_location=device)
    config = ClassifierConfig(**checkpoint['config']['model_config'])
    consistency_model = MultimodalClassifier(config)
    consistency_model.load_state_dict(checkpoint['model_state_dict'])
    consistency_model.to(device)
    print("Loaded consistency-aware model")
else:
    print("Checkpoint not found!")
    consistency_model = None

In [ ]:
# Load standard model
standard_checkpoint = CHECKPOINT_DIR / 'standard' / 'standard_best.pt'
if standard_checkpoint.exists():
    checkpoint = torch.load(standard_checkpoint, map_location=device)
    config = ClassifierConfig(**checkpoint['config']['model_config'])
    standard_model = MultimodalClassifier(config)
    standard_model.load_state_dict(checkpoint['model_state_dict'])
    standard_model.to(device)
    print("Loaded standard model")
else:
    print("Standard model checkpoint not found")
    standard_model = None

## Create Image-Swapped Dataset

In [ ]:
# Configure swap experiment
swap_config = SwapConfig(
    num_swaps=1000,
    seed=42,
    replacement_strategy='random'
)

# Create swap experiment
swap_experiment = ImageSwapExperiment(
    dataset=test_dataset,
    config=swap_config,
    image_pool_dir=str(IMAGES_DIR)
)

# Create swapped dataset
swapped_dataset = swap_experiment.create_swapped_dataset(num_samples=min(1000, len(test_dataset)))
swapped_loader = DataLoader(swapped_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Swapped dataset: {len(swapped_dataset)} samples")

## Run Swap Experiment on Consistency-Aware Model

In [ ]:
if consistency_model:
    print("Running swap experiment on consistency-aware model...")
    
    consistency_swap_results = controlled_swap_evaluation(
        model=consistency_model,
        preprocessor=preprocessor,
        original_loader=test_loader,
        swapped_loader=swapped_loader,
        device=device
    )
    
    print("\nResults:")
    print(f"Original accuracy: {consistency_swap_results['original_metrics']['accuracy']:.4f}")
    print(f"Swapped accuracy: {consistency_swap_results['swapped_metrics']['accuracy']:.4f}")
    print(f"Difference: {consistency_swap_results['differences']['accuracy']:.4f}")
    
    print(f"\nPrediction flips: {consistency_swap_results['prediction_flips']['total_flips']}")
    print(f"Flip rate: {consistency_swap_results['prediction_flips']['flip_rate']:.2%}")

## Run Swap Experiment on Standard Model

In [ ]:
if standard_model:
    print("Running swap experiment on standard model...")
    
    standard_swap_results = controlled_swap_evaluation(
        model=standard_model,
        preprocessor=preprocessor,
        original_loader=test_loader,
        swapped_loader=swapped_loader,
        device=device
    )
    
    print("\nResults:")
    print(f"Original accuracy: {standard_swap_results['original_metrics']['accuracy']:.4f}")
    print(f"Swapped accuracy: {standard_swap_results['swapped_metrics']['accuracy']:.4f}")
    print(f"Difference: {standard_swap_results['differences']['accuracy']:.4f}")
    
    print(f"\nPrediction flips: {standard_swap_results['prediction_flips']['total_flips']}")
    print(f"Flip rate: {standard_swap_results['prediction_flips']['flip_rate']:.2%}")

## Compare Models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))

models = ['Consistency-Aware', 'Standard']
original_acc = [
    consistency_swap_results['original_metrics']['accuracy'] if consistency_model else 0,
    standard_swap_results['original_metrics']['accuracy'] if standard_model else 0
]
swapped_acc = [
    consistency_swap_results['swapped_metrics']['accuracy'] if consistency_model else 0,
    standard_swap_results['swapped_metrics']['accuracy'] if standard_model else 0
]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, original_acc, width, label='Original', color='blue')
bars2 = ax.bar(x + width/2, swapped_acc, width, label='Image-Swapped', color='red')

ax.set_ylabel('Accuracy')
ax.set_title('Original vs Image-Swapped Performance')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim([0, 1])

# Add values
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{bar.get_height():.3f}', ha='center', va='bottom')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{bar.get_height():.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'swap_experiment_comparison.png', dpi=150)
plt.show()

## Save Results

In [ ]:
from src.utils import save_json

swap_results = {
    'consistency_aware': consistency_swap_results if consistency_model else None,
    'standard': standard_swap_results if standard_model else None,
    'config': swap_config.__dict__
}

save_json(swap_results, RESULTS_DIR / 'image_swap_results.json')

print("Results saved to results/image_swap_results.json")

## Analysis Summary

In [ ]:
print("\n" + "="*50)
print("IMAGE-SWAP EXPERIMENT SUMMARY")
print("="*50)

print("\nThis experiment tests how models respond when images are replaced with")
print("unrelated images while keeping text and original labels the same.")

print("\nKey findings:")

if consistency_model and standard_model:
    ca_drop = consistency_swap_results['original_metrics']['accuracy'] - consistency_swap_results['swapped_metrics']['accuracy']
    st_drop = standard_swap_results['original_metrics']['accuracy'] - standard_swap_results['swapped_metrics']['accuracy']
    
    print(f"  Consistency-Aware accuracy drop: {ca_drop:.4f}")
    print(f"  Standard model accuracy drop: {st_drop:.4f}")
    
    if ca_drop < st_drop:
        print("\n  ✓ Consistency-aware model is MORE robust to image swaps")
    else:
        print("\n  ✗ Standard model is MORE robust to image swaps")

print("\nNote: A model that responds strongly to image swaps may be")
print("over-relying on visual features rather than semantic consistency.")

print("\nNext: Run 06_evaluation_and_plots.ipynb for comprehensive evaluation")